In [0]:
import os, sys, subprocess

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"
DBT_LIB = "/tmp/dbtlib"
PROFILES = "/tmp/dbt_profiles"

# 1. Install pinned dbt into its own folder (keeps notebook packages untouched)
if not os.path.exists(f"{DBT_LIB}/dbt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--ignore-installed",
                    "--target", DBT_LIB, "dbt-core==1.12.5", "dbt-databricks==1.12.7"], check=True)

# 2. Connection settings (token from the secret scope, never in the repo)
os.makedirs(PROFILES, exist_ok=True)
with open(f"{PROFILES}/profiles.yml", "w") as f:
    f.write("""oil_shock:
  target: prod
  outputs:
    prod:
      type: databricks
      catalog: oil_shock
      schema: silver
      host: dbc-050dc881-f83e.cloud.databricks.com
      http_path: /sql/1.0/warehouses/42944d45ee999e1c
      token: "{{ env_var('DBT_DATABRICKS_TOKEN') }}"
      threads: 4
""")

env = dict(os.environ,
           PYTHONPATH=DBT_LIB, PYTHONNOUSERSITE="1",
           DBT_DATABRICKS_TOKEN=dbutils.secrets.get("oil-shock", "databricks-token"),
           DBT_TARGET_PATH="/tmp/dbt_target", DBT_LOG_PATH="/tmp/dbt_logs",
           DBT_PROFILES_DIR=PROFILES)

# 3. Build everything (seeds, models, tests)
r = subprocess.run([sys.executable, "-c", "from dbt.cli.main import cli; cli()",
                    "build", "--project-dir", DBT],
                   capture_output=True, text=True, env=env)
print(r.stdout[-8000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("dbt build failed — see output above")
print("dbt build OK")